# Phase 6: RAG & Vector Databases
## Day 27: ChunkingStrategies

Date: 2026-04-24

### Learning objectives
- Understand why chunking matters in RAG.
- Create fixed-size chunks.
- Add overlap between chunks.
- Build a recursive character splitter.
- Try simple semantic-style chunking.
- Compare how chunking affects retrieval quality.

In [ ]:
import json
import re
import textwrap
from pprint import pprint

import numpy as np
import pandas as pd

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    SKLEARN_AVAILABLE = True
except Exception:
    TfidfVectorizer = None
    SKLEARN_AVAILABLE = False

def show(title, content):
    print("\n" + "=" * 82)
    print(title)
    print("=" * 82)
    print(textwrap.dedent(str(content)).strip())

np.set_printoptions(precision=4, suppress=True)

print("Setup complete.")
print("scikit-learn available:", SKLEARN_AVAILABLE)

In [ ]:
source_documents = [
    {
        "doc_id": "DOC-CAMPAIGN",
        "title": "Campaign Performance Notes",
        "text": '''
        Spring Coffee Push was an Instagram campaign for Berlin customers.
        The campaign spent 1200 EUR and produced 3420 clicks.
        It generated 184 conversions after a limited-time discount was added.
        The conversion rate was strong compared with previous coffee campaigns.
        The next recommendation is to scale the campaign carefully and monitor cost per conversion.

        Bank App Onboarding was an email campaign for existing banking users.
        The campaign spent 800 EUR and produced 980 clicks.
        It generated only 42 conversions.
        The subject line was too generic and the call to action was not clear.
        The next recommendation is to rewrite the subject line and test a simpler onboarding message.

        Yoga Studio Trial was a TikTok campaign for beginners in Berlin.
        It spent 650 EUR and produced 2100 clicks.
        It generated 165 conversions.
        Short videos with beginner-friendly copy performed best.
        The next recommendation is to create more short videos and test new landing page copy.
        '''
    },
    {
        "doc_id": "DOC-OCR",
        "title": "OCR Pipeline Notes",
        "text": '''
        The OCR pipeline starts with a document image.
        OpenCV preprocessing improves OCR quality with grayscale conversion, denoising, thresholding, and deskewing.
        Tesseract works well on clean scans and simple layouts.
        EasyOCR can be useful for natural images and multilingual text.
        After OCR, raw text should be cleaned before structured extraction.

        The LLM extraction step receives OCR text and a JSON schema.
        The prompt should request valid JSON only.
        The extracted record should be validated with field types and business rules.
        If JSON parsing fails, the pipeline can repair small formatting issues.
        If validation fails, the pipeline can retry with the error message.
        '''
    },
    {
        "doc_id": "DOC-RAG",
        "title": "RAG Design Notes",
        "text": '''
        A RAG system retrieves relevant document chunks before generating an answer.
        Chunking strategy affects retrieval quality, answer accuracy, and context length.
        Very large chunks can contain too many ideas and reduce precision.
        Very small chunks can lose important context.
        Overlap helps preserve information that appears near chunk boundaries.

        Embeddings turn text chunks into vectors.
        A vector database stores those embeddings and supports similarity search.
        Retrieval quality should be evaluated with test queries and expected documents.
        Bad retrieval usually causes weak answers even if the language model is strong.
        Good RAG systems keep source metadata so answers can cite the right document.
        '''
    }
]

for doc in source_documents:
    print(doc["doc_id"], "-", doc["title"], "-", len(doc["text"]), "characters")

## 1. Why chunking matters

RAG does not usually embed a whole long document as one vector.

It splits documents into chunks. Good chunks keep enough context while staying focused.

In [ ]:
rag_pipeline = pd.DataFrame([
    {"step": 1, "stage": "Load documents", "output": "Raw text and metadata"},
    {"step": 2, "stage": "Chunk documents", "output": "Smaller text pieces"},
    {"step": 3, "stage": "Embed chunks", "output": "Vector for each chunk"},
    {"step": 4, "stage": "Store vectors", "output": "Vector database"},
    {"step": 5, "stage": "Retrieve", "output": "Top matching chunks for a query"},
    {"step": 6, "stage": "Generate", "output": "Answer using retrieved context"},
])

rag_pipeline

In [ ]:
def chunk_stats(chunks):
    lengths = [len(chunk["text"]) for chunk in chunks]
    return {
        "num_chunks": len(chunks),
        "min_chars": min(lengths) if lengths else 0,
        "max_chars": max(lengths) if lengths else 0,
        "avg_chars": round(float(np.mean(lengths)), 1) if lengths else 0,
    }

example_chunk = {
    "chunk_id": "DOC-RAG-000",
    "doc_id": "DOC-RAG",
    "text": "A RAG system retrieves relevant document chunks before generating an answer.",
    "metadata": {"title": "RAG Design Notes"}
}

pprint(example_chunk)

## 2. Fixed-size character chunking

Fixed-size chunking cuts text every N characters.

It is simple, but it can split sentences in awkward places.

In [ ]:
def fixed_size_chunks(text, chunk_size=300):
    chunks = []

    for start in range(0, len(text), chunk_size):
        end = start + chunk_size
        chunks.append(text[start:end].strip())

    return [chunk for chunk in chunks if chunk]

sample_text = source_documents[2]["text"]
fixed_chunks = fixed_size_chunks(sample_text, chunk_size=260)

for i, chunk in enumerate(fixed_chunks):
    print(f"Chunk {i} | {len(chunk)} chars")
    print(chunk[:180].replace("\n", " "))
    print()

In [ ]:
def build_chunk_records(docs, chunking_function, strategy_name, **kwargs):
    records = []

    for doc in docs:
        chunks = chunking_function(doc["text"], **kwargs)

        for index, chunk_text in enumerate(chunks):
            records.append({
                "chunk_id": f'{doc["doc_id"]}-{strategy_name}-{index:03d}',
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "strategy": strategy_name,
                "chunk_index": index,
                "text": chunk_text
            })

    return records

fixed_records = build_chunk_records(
    source_documents,
    fixed_size_chunks,
    "fixed",
    chunk_size=260
)

print("Fixed chunk stats:")
pprint(chunk_stats(fixed_records))
pd.DataFrame(fixed_records).head()

## 3. Chunk overlap

Overlap repeats some text from the previous chunk.

This helps when important facts sit near chunk boundaries.

In [ ]:
def fixed_size_chunks_with_overlap(text, chunk_size=300, overlap=60):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

overlap_chunks = fixed_size_chunks_with_overlap(sample_text, chunk_size=260, overlap=70)

for i, chunk in enumerate(overlap_chunks):
    print(f"Chunk {i} | {len(chunk)} chars")
    print(chunk[:180].replace("\n", " "))
    print()

In [ ]:
overlap_records = build_chunk_records(
    source_documents,
    fixed_size_chunks_with_overlap,
    "overlap",
    chunk_size=260,
    overlap=70
)

print("Overlap chunk stats:")
pprint(chunk_stats(overlap_records))
pd.DataFrame(overlap_records).head()

## 4. Sentence-based chunking

Sentence chunking tries to avoid splitting sentences.

This usually creates more readable chunks than raw character splitting.

In [ ]:
def split_sentences(text):
    clean = re.sub(r"\s+", " ", text.strip())
    sentences = re.split(r"(?<=[.!?])\s+", clean)
    return [sentence.strip() for sentence in sentences if sentence.strip()]

sentences = split_sentences(source_documents[0]["text"])

for i, sentence in enumerate(sentences[:5]):
    print(i, sentence)

In [ ]:
def sentence_chunks(text, max_chars=320):
    sentences = split_sentences(text)
    chunks = []
    current = ""

    for sentence in sentences:
        candidate = (current + " " + sentence).strip()

        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current)
            current = sentence

    if current:
        chunks.append(current)

    return chunks

sentence_based_chunks = sentence_chunks(source_documents[0]["text"], max_chars=320)

for i, chunk in enumerate(sentence_based_chunks):
    print(f"Chunk {i} | {len(chunk)} chars")
    print(chunk)
    print()

In [ ]:
sentence_records = build_chunk_records(
    source_documents,
    sentence_chunks,
    "sentence",
    max_chars=320
)

print("Sentence chunk stats:")
pprint(chunk_stats(sentence_records))
pd.DataFrame(sentence_records).head()

## 5. Recursive character splitter

A recursive splitter tries larger separators first.

It prefers paragraphs, then sentences, then spaces, then characters.

In [ ]:
def recursive_split_text(text, chunk_size=320, separators=None):
    separators = separators or ["\n\n", "\n", ". ", " ", ""]
    text = text.strip()

    if len(text) <= chunk_size:
        return [text] if text else []

    separator = separators[0]
    remaining_separators = separators[1:]

    if separator == "":
        return [text[i:i + chunk_size].strip() for i in range(0, len(text), chunk_size) if text[i:i + chunk_size].strip()]

    parts = text.split(separator)

    if len(parts) == 1:
        return recursive_split_text(text, chunk_size=chunk_size, separators=remaining_separators)

    chunks = []
    current = ""

    for part in parts:
        if not part.strip():
            continue

        candidate = (current + separator + part).strip() if current else part.strip()

        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.extend(recursive_split_text(current, chunk_size=chunk_size, separators=remaining_separators))
            current = part.strip()

    if current:
        chunks.extend(recursive_split_text(current, chunk_size=chunk_size, separators=remaining_separators))

    return chunks

recursive_chunks = recursive_split_text(source_documents[1]["text"], chunk_size=320)

for i, chunk in enumerate(recursive_chunks):
    print(f"Chunk {i} | {len(chunk)} chars")
    print(chunk.replace("\n", " "))
    print()

In [ ]:
recursive_records = build_chunk_records(
    source_documents,
    recursive_split_text,
    "recursive",
    chunk_size=320
)

print("Recursive chunk stats:")
pprint(chunk_stats(recursive_records))
pd.DataFrame(recursive_records).head()

## 6. Semantic-style chunking

True semantic chunking often uses embeddings to detect topic shifts.

Here we use a simple local version. We split on paragraphs and keep paragraphs as chunks.

In [ ]:
def paragraph_chunks(text, max_chars=500):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []

    for paragraph in paragraphs:
        if len(paragraph) <= max_chars:
            chunks.append(re.sub(r"\s+", " ", paragraph))
        else:
            chunks.extend(sentence_chunks(paragraph, max_chars=max_chars))

    return chunks

semantic_chunks = paragraph_chunks(source_documents[0]["text"], max_chars=500)

for i, chunk in enumerate(semantic_chunks):
    print(f"Chunk {i} | {len(chunk)} chars")
    print(chunk)
    print()

In [ ]:
semantic_records = build_chunk_records(
    source_documents,
    paragraph_chunks,
    "semantic_style",
    max_chars=500
)

print("Semantic-style chunk stats:")
pprint(chunk_stats(semantic_records))
pd.DataFrame(semantic_records).head()

## 7. Add metadata to chunks

Chunks need metadata.

Metadata helps you filter, cite, debug, and trace answers back to source documents.

In [ ]:
def add_chunk_metadata(records):
    enriched = []

    for record in records:
        text = record["text"]
        enriched.append({
            **record,
            "char_count": len(text),
            "word_count": len(text.split()),
            "preview": text[:90].replace("\n", " ") + ("..." if len(text) > 90 else "")
        })

    return enriched

enriched_recursive_records = add_chunk_metadata(recursive_records)
pd.DataFrame(enriched_recursive_records).head()

In [ ]:
metadata_columns = ["chunk_id", "doc_id", "title", "strategy", "chunk_index", "char_count", "word_count", "preview"]
pd.DataFrame(enriched_recursive_records)[metadata_columns]

## 8. Compare chunking strategies

Chunk count and chunk length affect retrieval.

More chunks can improve precision but increase storage and search cost.

In [ ]:
all_strategy_records = {
    "fixed": fixed_records,
    "overlap": overlap_records,
    "sentence": sentence_records,
    "recursive": recursive_records,
    "semantic_style": semantic_records,
}

strategy_summary = []

for strategy, records in all_strategy_records.items():
    stats = chunk_stats(records)
    strategy_summary.append({
        "strategy": strategy,
        **stats
    })

pd.DataFrame(strategy_summary)

In [ ]:
def estimate_storage_for_chunks(num_chunks, dimensions=1536, bytes_per_value=4):
    total_bytes = num_chunks * dimensions * bytes_per_value
    return round(total_bytes / (1024 ** 2), 3)

storage_rows = []

for strategy, records in all_strategy_records.items():
    storage_rows.append({
        "strategy": strategy,
        "num_chunks": len(records),
        "estimated_mb_at_1536_dims": estimate_storage_for_chunks(len(records), dimensions=1536)
    })

pd.DataFrame(storage_rows)

## 9. Retrieval test

Now we test whether different chunking strategies return useful chunks.

We use TF-IDF for local retrieval so no API key is needed.

In [ ]:
def cosine_similarity(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)

def simple_token_embedding(text, vocabulary):
    text_lower = text.lower()
    return np.array([text_lower.count(term) for term in vocabulary], dtype=float)

def build_retriever(records):
    chunk_texts = [record["text"] for record in records]

    if SKLEARN_AVAILABLE:
        vectorizer = TfidfVectorizer(stop_words="english")
        matrix = vectorizer.fit_transform(chunk_texts).toarray()

        def embed_query(query):
            return vectorizer.transform([query]).toarray()[0]

        return matrix, embed_query

    vocabulary = sorted(set(re.findall(r"[a-z]+", " ".join(chunk_texts).lower())))
    matrix = np.vstack([simple_token_embedding(text, vocabulary) for text in chunk_texts])

    def embed_query(query):
        return simple_token_embedding(query, vocabulary)

    return matrix, embed_query

def search_chunks(records, query, top_k=3):
    matrix, embed_query = build_retriever(records)
    query_vector = embed_query(query)

    scores = [cosine_similarity(query_vector, row) for row in matrix]

    results = pd.DataFrame(records).copy()
    results["score"] = scores
    return results.sort_values("score", ascending=False).head(top_k)

search_chunks(recursive_records, "Which campaign had a generic subject line?", top_k=3)[["chunk_id", "doc_id", "score", "text"]]

In [ ]:
test_queries = [
    {"query": "Which campaign had a generic subject line?", "expected_doc_id": "DOC-CAMPAIGN"},
    {"query": "How does preprocessing improve OCR quality?", "expected_doc_id": "DOC-OCR"},
    {"query": "Why is overlap useful in chunking?", "expected_doc_id": "DOC-RAG"},
    {"query": "What is the recommendation for Yoga Studio Trial?", "expected_doc_id": "DOC-CAMPAIGN"}
]

def evaluate_strategy(records, strategy_name, test_queries, top_k=2):
    rows = []

    for test in test_queries:
        results = search_chunks(records, test["query"], top_k=top_k)
        retrieved_doc_ids = results["doc_id"].tolist()
        hit = test["expected_doc_id"] in retrieved_doc_ids

        rows.append({
            "strategy": strategy_name,
            "query": test["query"],
            "expected_doc_id": test["expected_doc_id"],
            "retrieved_doc_ids": retrieved_doc_ids,
            "hit": hit,
            "top_score": round(float(results.iloc[0]["score"]), 4)
        })

    return rows

evaluation_rows = []

for strategy, records in all_strategy_records.items():
    evaluation_rows.extend(evaluate_strategy(records, strategy, test_queries, top_k=2))

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df

In [ ]:
strategy_scores = (
    evaluation_df
    .groupby("strategy", as_index=False)
    .agg(hit_rate=("hit", "mean"), avg_top_score=("top_score", "mean"))
    .sort_values("hit_rate", ascending=False)
)

strategy_scores

## 10. Boundary problem demo

Chunk boundaries can hide useful context.

Overlap helps when a question needs words from both sides of a split.

In [ ]:
boundary_text = (
    "The refund policy says customers can request a voucher after a delivery delay. "
    "The voucher is only issued after successful delivery. "
    "This prevents users from claiming compensation and then cancelling the order."
)

no_overlap = fixed_size_chunks_with_overlap(boundary_text, chunk_size=95, overlap=0)
with_overlap = fixed_size_chunks_with_overlap(boundary_text, chunk_size=95, overlap=35)

print("No overlap:")
for i, chunk in enumerate(no_overlap):
    print(i, repr(chunk))

print("\nWith overlap:")
for i, chunk in enumerate(with_overlap):
    print(i, repr(chunk))

In [ ]:
def contains_terms(chunk, terms):
    chunk_lower = chunk.lower()
    return all(term.lower() in chunk_lower for term in terms)

terms = ["voucher", "successful delivery"]

print("No overlap chunks containing both terms:")
print([i for i, chunk in enumerate(no_overlap) if contains_terms(chunk, terms)])

print("With overlap chunks containing both terms:")
print([i for i, chunk in enumerate(with_overlap) if contains_terms(chunk, terms)])

## 11. Choosing chunk size and overlap

There is no perfect chunk size.

Start with a reasonable default, then evaluate retrieval on real questions.

In [ ]:
def recommend_chunking_strategy(document_type, query_type):
    document_type = document_type.lower()
    query_type = query_type.lower()

    if "faq" in document_type or "short" in document_type:
        return {"strategy": "paragraph or sentence chunks", "chunk_size": "200 to 500 chars", "overlap": "0 to 50 chars"}

    if "manual" in document_type or "long" in document_type:
        return {"strategy": "recursive splitter", "chunk_size": "600 to 1200 chars", "overlap": "100 to 200 chars"}

    if "exact" in query_type or "field" in query_type:
        return {"strategy": "small focused chunks", "chunk_size": "300 to 700 chars", "overlap": "50 to 100 chars"}

    return {"strategy": "recursive splitter", "chunk_size": "500 to 1000 chars", "overlap": "80 to 150 chars"}

scenarios = [
    ("short FAQ", "direct question"),
    ("long policy manual", "broad question"),
    ("invoice OCR reports", "exact field lookup"),
    ("mixed knowledge base", "general search"),
]

for document_type, query_type in scenarios:
    print("\nScenario:", document_type, "|", query_type)
    pprint(recommend_chunking_strategy(document_type, query_type))

## Tricky bits

Chunking mistakes are easy to miss.

A RAG answer can look weak because retrieval was weak, not because the LLM was weak.

In [ ]:
tricky_bits = pd.DataFrame([
    {"problem": "Chunks are too large", "symptom": "Retrieved context has many unrelated ideas", "fix": "Reduce chunk size or use semantic-style splitting"},
    {"problem": "Chunks are too small", "symptom": "Retrieved context misses key details", "fix": "Increase chunk size or add overlap"},
    {"problem": "No metadata", "symptom": "You cannot cite or debug sources", "fix": "Store doc_id, title, page, and chunk index"},
    {"problem": "No evaluation", "symptom": "You guess instead of measuring retrieval quality", "fix": "Create test queries with expected chunks"},
    {"problem": "Blind overlap", "symptom": "Too many duplicate chunks and high storage cost", "fix": "Use enough overlap, not maximum overlap"},
])

tricky_bits

In [ ]:
def diagnose_chunking_issue(symptom):
    symptom = symptom.lower()

    if "unrelated" in symptom or "too broad" in symptom:
        return "Chunks may be too large. Try smaller or semantic chunks."
    if "missing context" in symptom or "incomplete" in symptom:
        return "Chunks may be too small. Increase chunk size or overlap."
    if "duplicate" in symptom:
        return "Overlap may be too high. Reduce overlap."
    if "cannot cite" in symptom:
        return "Metadata is missing. Add doc_id, title, and chunk index."
    return "Review chunk size, overlap, separators, and retrieval examples."

symptoms = [
    "Retrieved chunks are unrelated",
    "Answer has missing context",
    "There are too many duplicate chunks",
    "The system cannot cite sources"
]

for symptom in symptoms:
    print(symptom, "=>", diagnose_chunking_issue(symptom))

## Trick questions

1. Is a larger chunk always better?

<details>
<summary>Answer</summary>

No. Large chunks can include too many topics and reduce retrieval precision.

</details>

2. Why use overlap?

<details>
<summary>Answer</summary>

Overlap protects information that appears near chunk boundaries.

</details>

3. What is the downside of too much overlap?

<details>
<summary>Answer</summary>

It creates duplicate content, increases storage, and can make retrieval repetitive.

</details>

4. Why keep metadata with each chunk?

<details>
<summary>Answer</summary>

Metadata lets you cite sources, filter results, and debug retrieval.

</details>

5. What is the best way to choose chunk size?

<details>
<summary>Answer</summary>

Start with a reasonable default, then evaluate retrieval with real test queries.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Create fixed-size chunks from sample_text.

chunks = ___

assert isinstance(chunks, list)
assert len(chunks) > 1
assert all(isinstance(chunk, str) for chunk in chunks)
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Create overlapping chunks from sample_text.

overlap_result = ___

assert isinstance(overlap_result, list)
assert len(overlap_result) >= len(chunks)
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Split source document 0 into sentence chunks.

sentence_result = ___

assert isinstance(sentence_result, list)
assert len(sentence_result) > 0
assert all(len(chunk) <= 320 for chunk in sentence_result)
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Create recursive chunks from DOC-OCR text.

recursive_result = ___

assert isinstance(recursive_result, list)
assert len(recursive_result) > 0
assert all(len(chunk) <= 320 for chunk in recursive_result)
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Build chunk records using recursive splitting.

records = ___

assert isinstance(records, list)
assert len(records) > 0
assert {"chunk_id", "doc_id", "text"}.issubset(records[0].keys())
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Add metadata to the records.

records_with_metadata = ___

assert "char_count" in records_with_metadata[0]
assert "word_count" in records_with_metadata[0]
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Search recursive chunks for an OCR query.

results = ___

assert isinstance(results, pd.DataFrame)
assert len(results) == 3
assert "score" in results.columns
print("Exercise 7 passed.")

In [ ]:
# Exercise 8
# Evaluate the recursive strategy.

eval_rows = ___

assert isinstance(eval_rows, list)
assert len(eval_rows) == len(test_queries)
assert "hit" in eval_rows[0]
print("Exercise 8 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
chunks = fixed_size_chunks(sample_text, chunk_size=260)
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
overlap_result = fixed_size_chunks_with_overlap(sample_text, chunk_size=260, overlap=70)
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
sentence_result = sentence_chunks(source_documents[0]["text"], max_chars=320)
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
recursive_result = recursive_split_text(source_documents[1]["text"], chunk_size=320)
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
records = build_chunk_records(
    source_documents,
    recursive_split_text,
    "recursive",
    chunk_size=320
)
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
records_with_metadata = add_chunk_metadata(records)
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
results = search_chunks(recursive_records, "OCR preprocessing quality", top_k=3)
```

</details>

<details>
<summary>Exercise 8 solution</summary>

```python
eval_rows = evaluate_strategy(recursive_records, "recursive", test_queries, top_k=2)
```

</details>

## Cumulative review exercises

These mix topics from Days 17 to 26. Fill in `___` and run each cell.

In [ ]:
# Review 1: Ollama
# Fill the default local generate endpoint.

ollama_url = ___

assert ollama_url == "http://localhost:11434/api/generate"
print("Review 1 passed.")

In [ ]:
# Review 2: Prompt engineering
# Choose the prompting style that uses examples.

prompt_style = ___

assert prompt_style.lower() == "few-shot"
print("Review 2 passed.")

In [ ]:
# Review 3: Structured output
# Parse JSON text.

json_text = '{"campaign": "Demo", "clicks": 100}'
parsed = ___

assert parsed["clicks"] == 100
print("Review 3 passed.")

In [ ]:
# Review 4: Information extraction
# Calculate conversion rate.

record = {"clicks": 1000, "conversions": 80}
conversion_rate = ___

assert abs(conversion_rate - 0.08) < 1e-9
print("Review 4 passed.")

In [ ]:
# Review 5: Tesseract basics
# Choose the language code for English.

english_lang_code = ___

assert english_lang_code == "eng"
print("Review 5 passed.")

In [ ]:
# Review 6: EasyOCR
# Create a language list for English and German.

easyocr_languages = ___

assert easyocr_languages == ["en", "de"] or easyocr_languages == ["de", "en"]
print("Review 6 passed.")

In [ ]:
# Review 7: OpenCV preprocessing
# Choose the thresholding method useful for uneven lighting.

threshold_method = ___

assert threshold_method.lower() == "adaptive"
print("Review 7 passed.")

In [ ]:
# Review 8: OCR plus LLM pipeline
# Choose the structured output format.

structured_format = ___

assert structured_format.upper() == "JSON"
print("Review 8 passed.")

In [ ]:
# Review 9: Document intelligence
# Create a quality rule for invoices.

quality_rule = ___

assert "total" in quality_rule.lower() or "date" in quality_rule.lower() or "id" in quality_rule.lower()
print("Review 9 passed.")

In [ ]:
# Review 10: Embeddings
# Calculate cosine similarity.

a = np.array([1, 0, 0])
b = np.array([1, 1, 0])
score = ___

assert 0.70 < score < 0.72
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
ollama_url = "http://localhost:11434/api/generate"

# Review 2
prompt_style = "few-shot"

# Review 3
parsed = json.loads(json_text)

# Review 4
conversion_rate = record["conversions"] / record["clicks"]

# Review 5
english_lang_code = "eng"

# Review 6
easyocr_languages = ["en", "de"]

# Review 7
threshold_method = "adaptive"

# Review 8
structured_format = "JSON"

# Review 9
quality_rule = "Invoice total must be present and non-negative."

# Review 10
score = cosine_similarity(a, b)
```

</details>

In [ ]:
cheat_sheet = '''
DAY 27 CHEAT SHEET: CHUNKING STRATEGIES

Why chunking matters:
- RAG retrieves chunks, not usually full documents.
- Good chunks improve retrieval and answer quality.
- Bad chunks can make a strong LLM look weak.

Strategies:
- Fixed-size chunking: simple but may split sentences.
- Overlap chunking: protects boundary context.
- Sentence chunking: keeps sentences intact.
- Recursive splitting: tries paragraphs, lines, sentences, spaces, then characters.
- Semantic-style chunking: keeps topic-based sections together.

Metadata to store:
- chunk_id
- doc_id
- title
- chunk_index
- page number if available
- strategy
- char_count and word_count

Tuning tips:
- Too large: lower precision.
- Too small: missing context.
- Too much overlap: duplicates and higher cost.
- No overlap: boundary context can be lost.
- Always evaluate retrieval with test queries.
'''

print(cheat_sheet)

## Next up: Day 28 — ChromaAndFAISS

You will learn ChromaDB collections, FAISS index types, and similarity search.